In [33]:
# Cell 1 — run this first, before anything else
import sys
# Remove all sentinel modules so they reload fresh
keys_to_remove = [k for k in sys.modules if 'sentinel' in k or 'src' in k]
for k in keys_to_remove:
    del sys.modules[k]

In [34]:
from src.retrieval.retriever import (
    _vector_search, _fulltext_search,
    _reciprocal_rank_fusion, _rerank
)
from sentence_transformers import SentenceTransformer
import psycopg2, os
from dotenv import load_dotenv

Loading retrieval models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieval models ready.


In [35]:
load_dotenv()

EMBED_MODEL = SentenceTransformer("BAAI/bge-base-en-v1.5")
CONN        = psycopg2.connect(os.getenv("DATABASE_URL"))

query  = "What are Apple's cybersecurity risks?"
ticker = "AAPL"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [36]:
# Step 1 — see raw vector results
qvec   = EMBED_MODEL.encode(query, normalize_embeddings=True).tolist()
v_res  = _vector_search(qvec, ticker, 10, CONN)
print("=== Vector search top 5 ===")
for r in v_res[:5]:
    print(f"  score={r['vector_score']:.4f} | {r['section']} | {r['text'][:80]}...")


=== Vector search top 5 ===
  score=0.7218 | item_1a | Apple Inc. | 2025 Form 10-K | 11
As with all companies, the security the Company...
  score=0.6771 | item_1a | The Company’s business also requires it to share confidential information with s...
  score=0.6622 | item_1a | Legal and Regulatory Compliance Risks
The Company’s business, results of operati...
  score=0.6147 | item_1a | Apple Inc. | 2025 Form 10-K | 6
Following any interruption to its business, the ...
  score=0.6100 | item_1a | Adverse economic conditions can also lead to increased credit and collectibility...


In [37]:
# Step 2 — see raw FTS results
f_res  = _fulltext_search(query, ticker, 10, CONN)
print("\n=== Full-text search top 5 ===")
print(f_res)
for r in f_res[:5]:
    print(f"score={r['fts_score']:.5f} | {r['section']} | {r['text'][:80]}...")


=== Full-text search top 5 ===
[{'id': 1872, 'ticker': 'AAPL', 'filing_date': '2025-10-31', 'section': 'item_1a', 'chunk_index': 4, 'text': 'Many of the Company’s operations, retail stores and facilities, as well as critical business operations of the Company’s suppliers and contract manufacturers, are in locations that are prone to earthquakes and other natural disasters. Global climate change is resulting in certain types of natural disasters and extreme weather occurring more frequently or with more intense effects. In addition, the Company’s and its suppliers’ operations, retail stores and facilities are subject to the risk of interruption by fire, power shortages, nuclear power plant accidents and other industrial accidents, terrorist attacks and other hostile acts, ransomware and other cybersecurity attacks, labor disputes, public health issues and other events beyond the Company’s control.\nSuch events can make it difficult or impossible for the Company to manufacture and deliv

In [38]:
# Step 3 — see what RRF changes
merged = _reciprocal_rank_fusion(v_res, f_res)
print("\n=== After RRF merge top 5 ===")
for r in merged[:5]:
    print(f"  rrf={r['rrf_score']:.6f} | {r['section']} | {r['text'][:80]}...")


=== After RRF merge top 5 ===
  rrf=0.030886 | item_1a | Many of the Company’s operations, retail stores and facilities, as well as criti...
  rrf=0.016393 | item_1a | Apple Inc. | 2025 Form 10-K | 11
As with all companies, the security the Company...
  rrf=0.016129 | item_1a | The Company’s business also requires it to share confidential information with s...
  rrf=0.015873 | item_1a | Legal and Regulatory Compliance Risks
The Company’s business, results of operati...
  rrf=0.015625 | item_1a | Apple Inc. | 2025 Form 10-K | 6
Following any interruption to its business, the ...


In [39]:
# Step 4 — see what reranking changes
final  = _rerank(query, merged[:10], 5)
print("\n=== After reranking top 5 ===")
for r in final:
    print(f"  rerank={r['rerank_score']:.4f} | {r['section']} | {r['text'][:80]}...")


=== After reranking top 5 ===
  rerank=0.8511 | item_1a | Apple Inc. | 2025 Form 10-K | 11
As with all companies, the security the Company...
  rerank=0.7595 | item_1a | Many of the Company’s operations, retail stores and facilities, as well as criti...
  rerank=-1.0559 | item_1a | The Company’s business also requires it to share confidential information with s...
  rerank=-1.7059 | item_1a | Legal and Regulatory Compliance Risks
The Company’s business, results of operati...
  rerank=-2.2047 | item_1a | Adverse economic conditions can also lead to increased credit and collectibility...
